In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('src')
from config.paths import config
from config.constants import TARGET_COLUMN, NON_MEDICAL_FEATURES

In [ ]:
df = pd.read_csv(config.cleaned_data_file)
print(f"Loaded dataset shape: {df.shape}")
print(f"Target distribution:\n{df[TARGET_COLUMN].value_counts()}")

In [ ]:
print("DATA VALIDATION CHECKS")
print("-" * 40)

def validate_data(df):
    validation_results = {}

    validation_results['total_rows'] = len(df)
    validation_results['total_columns'] = len(df.columns)
    validation_results['missing_values'] = df.isnull().sum().sum()

    validation_results['duplicate_rows'] = df.duplicated().sum()

    validation_results['target_distribution'] = df[TARGET_COLUMN].value_counts().to_dict()

    numerical_cols = df.select_dtypes(include=[np.number]).columns
    validation_results['numerical_outliers'] = {}
    for col in numerical_cols:
        if col != TARGET_COLUMN:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
            validation_results['numerical_outliers'][col] = outliers

    categorical_cols = df.select_dtypes(include=['object']).columns
    validation_results['high_cardinality'] = {}
    for col in categorical_cols:
        if df[col].nunique() > 20:
            validation_results['high_cardinality'][col] = df[col].nunique()

    return validation_results

validation_before = validate_data(df)
print("Validation Results - Before Preprocessing:")
print(f"Total rows: {validation_before['total_rows']}")
print(f"Total columns: {validation_before['total_columns']}")
print(f"Missing values: {validation_before['missing_values']}")
print(f"Duplicate rows: {validation_before['duplicate_rows']}")
print(f"Target distribution: {validation_before['target_distribution']}")

if validation_before['duplicate_rows'] > 0:
    df = df.drop_duplicates()
    print(f"Removed {validation_before['duplicate_rows']} duplicate rows")

In [ ]:
print("1. FEATURE SELECTION STRATEGY")
print("-" * 40)

medical_features_removed = [col for col in df.columns if col not in NON_MEDICAL_FEATURES and col != TARGET_COLUMN]
print(f"Medical features removed: {len(medical_features_removed)}")
for feature in medical_features_removed:
    print(f"  - {feature}: Excluded - medical feature")

df_selected = df[NON_MEDICAL_FEATURES + [TARGET_COLUMN]].copy()
print(f"After medical feature removal: {df_selected.shape}")

numerical_features = df_selected.select_dtypes(include=[np.number]).columns.tolist()
numerical_features = [f for f in numerical_features if f != TARGET_COLUMN]

correlation_matrix = df_selected[numerical_features].corr()

high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

features_to_remove = set()
for feat1, feat2, corr in high_corr_pairs:
    feat1_na = df_selected[feat1].isna().sum()
    feat2_na = df_selected[feat2].isna().sum()

    if feat1_na > feat2_na:
        features_to_remove.add(feat1)
        print(f"  - {feat1}: Removed - high correlation ({corr:.3f}) with {feat2}, more missing values")
    else:
        features_to_remove.add(feat2)
        print(f"  - {feat2}: Removed - high correlation ({corr:.3f}) with {feat1}, more missing values")

df_selected = df_selected.drop(columns=features_to_remove)

low_variance_features = []
for col in df_selected.columns:
    if col != TARGET_COLUMN:
        unique_ratio = df_selected[col].nunique() / len(df_selected)
        if unique_ratio < 0.05:
            low_variance_features.append(col)
            print(f"  - {col}: Removed - low variance ({unique_ratio:.3f} unique ratio)")

df_selected = df_selected.drop(columns=low_variance_features)

X_temp = df_selected.drop(columns=[TARGET_COLUMN])
y_temp = df_selected[TARGET_COLUMN]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_temp.fillna(X_temp.median()), y_temp)

feature_importance = pd.DataFrame({
    'feature': X_temp.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

top_features = feature_importance.head(20)['feature'].tolist()
print(f"Top 20 features selected by importance:")
for i, (_, row) in enumerate(feature_importance.head(20).iterrows(), 1):
    print(f"  {i:2d}. {row['feature']}: {row['importance']:.4f}")

df_final_selected = df_selected[top_features + [TARGET_COLUMN]]
print(f"Final selected features: {df_final_selected.shape}")